<a href="https://colab.research.google.com/github/mrudulabankar1111/Mango-leaf-disease-detection/blob/main/mangoleafdiseasedetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from zipfile import ZipFile
import os

# Mount Drive if necessary
from google.colab import drive
drive.mount('/content/drive')

# Paths
zip_path = "/content/drive/MyDrive/mango_dataset.zip"
extract_path = "/content/dataset"

# Extract if not already
if not os.path.exists(extract_path):
    with ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

os.listdir(extract_path)
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (128, 128)
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = train_datagen.flow_from_directory(
    extract_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    extract_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False  # Important for matching predictions
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(train_data.class_indices), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(train_data, validation_data=val_data, epochs=5)
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

# Get true labels and predictions
val_preds = model.predict(val_data, verbose=1)
val_pred_classes = np.argmax(val_preds, axis=1)
true_classes = val_data.classes
class_labels = list(val_data.class_indices.keys())

# Classification report
report = classification_report(true_classes, val_pred_classes, target_names=class_labels, output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Store per-class metrics
class_metrics = {
    label: {
        'precision': round(report[label]['precision'] * 100, 2),
        'recall': round(report[label]['recall'] * 100, 2),
        'f1-score': round(report[label]['f1-score'] * 100, 2)
    } for label in class_labels
}

# Global accuracy
overall_accuracy = round(report['accuracy'] * 100, 2)
from google.colab import files
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

uploaded = files.upload()

for img_name in uploaded.keys():
    img_path = img_name
    img = image.load_img(img_path, target_size=(128,128))
    img_array = image.img_to_array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    class_index = np.argmax(prediction)
    class_labels = list(train_data.class_indices.keys())
    predicted_label = class_labels[class_index]

    plt.imshow(image.load_img(img_path))
    plt.title(f"Predicted: {predicted_label}")
    plt.axis('off')
    plt.show()
!pip install gradio --quiet

import gradio as gr
from PIL import Image

# Gradio prediction function
def predict_leaf(img: Image.Image):
    img = img.resize((128,128))
    img_array = np.array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)
    class_index = np.argmax(prediction)
    class_labels = list(train_data.class_indices.keys())
    predicted_label = class_labels[class_index]
    confidence = float(np.max(prediction))

    # Get metrics for predicted class
    metrics = class_metrics[predicted_label]
    precision = metrics['precision']
    recall = metrics['recall']
    f1 = metrics['f1-score']

    result = {
        "Predicted Disease": predicted_label,
        "Confidence (%)": round(confidence * 100, 2),
        "Precision (%)": precision,
        "Recall (%)": recall,
        "F1 Score (%)": f1,
        "Overall Accuracy (%)": overall_accuracy
    }

    return result

# Launch Gradio interface
interface = gr.Interface(
    fn=predict_leaf,
    inputs=gr.Image(type="pil"),
    outputs=gr.JSON(),
    title="🥭🍃 Mango Leaf Disease Detector",
    description="Upload a mango leaf image to predict its disease and view model performance metrics for the predicted class."
)

interface.launch(share=True)
{
  "Predicted Disease": "Bacterial_Canker",
  "Confidence (%)": 98.32,
  "Precision (%)": 91.67,
  "Recall (%)": 89.47,
  "F1 Score (%)": 90.56,
  "Overall Accuracy (%)": 92.5
}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 3200 images belonging to 8 classes.
Found 800 images belonging to 8 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 57600)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,372,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,393,352 (28.20 MB)

 Trainable params: 7,393,352 (28.20 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


100/100 ━━━━━━━━━━━━━━━━━━━━ 87s 852ms/step - accuracy: 0.2891 - loss: 2.1641 - val_accuracy: 0.5450 - val_loss: 1.1408
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 88s 877ms/step - accuracy: 0.6939 - loss: 0.8622 - val_accuracy: 0.6463 - val_loss: 0.9256
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 141s 870ms/step - accuracy: 0.8255 - loss: 0.5287 - val_accuracy: 0.6837 - val_loss: 0.8709
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 83s 829ms/step - accuracy: 0.8760 - loss: 0.3502 - val_accuracy: 0.7312 - val_loss: 0.7691
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 87s 868ms/step - accuracy: 0.9139 - loss: 0.2626 - val_accuracy: 0.7912 - val_loss: 0.6104
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 199ms/step
